# Latency & Accuracy analysis: side camera vs tracking vs motor commands

This notebook compares the three things the **side camera** sees against the
ground-truth logs of the experiment, and reports, **per finger**, the accuracy
and latency of (a) the visual **detection**, (b) the **tactor** movement, and
(c) the **motors**.

## The rig in one image
The side camera is framed so a single frame contains three functional regions:
```
+-------------------------------+--------------------+
|  VISION  (experiment monitor: |                    |
|  black bg, orange/blue object |     empty white    |
|  square, green progress bar)  |     desk           |
+----------------+--------------+                    |
|  MOTORS        |  TACTOR (green body, yellow centre,|
|  white box,    |  sitting on the participant finger)|
|  ~3 spools w/  |                                    |
|  a black line  |                                    |
+----------------+------------------------------------+
```
We therefore *separate* the video into those regions (`side_camera_separator.py`),
extract a signal from each (`region_signals.py`), and correlate them against
`tracking.csv` (the simulation ground truth) and `motor_commands.txt` (the ESP32
bridge log).

## Data reality / caveats (read me)
* The four pairs map 1:1 to the four tested fingers (configuration.csv):
  `pair_001=index, pair_002=middle, pair_003=ring, pair_004=pinky`, all at
  stiffness 145.
* **The `motor_commands.txt` log only covers ~15:30:48-15:31:32**, which is
  *after* every pair's video ends (the log was started late - the user noted
  only the *last part* exists). So motor commands do **not** temporally overlap
  the video for any finger. We therefore report the **firmware command->ack**
  latency (always computable from the log) and flag the missing overlap rather
  than inventing a number. The code will automatically compute the
  command->spool cross-correlation latency instead, if a future overlapping log
  is supplied.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the colocated helper modules importable.
HERE = Path.cwd()
if not (HERE / 'session_report.py').exists():
    # allow running from the repo root
    HERE = Path('validetion/latencyNaccurecy').resolve()
sys.path.insert(0, str(HERE))

import latency_analysis as LA
import session_report as SR
from motor_commands import parse_motor_commands
from build_signals import build

SESSION = HERE / 'Data' / '2026_06_28_18_42_single_finger_config_fingers_motor_set_0_123'
OUT = HERE / 'Results' / '2026_06_28_18_42_single_finger_config_fingers_motor_set_0_123' / 'signals'
print('session:', SESSION.name)
print('cached signals present:', (OUT / 'sessions_meta.csv').exists())

## 1. Region separation

**Why these algorithms** (see `side_camera_separator.py` for full notes):
* **Vision/monitor** = the largest *dark* blob (threshold + largest contour).
  Robust to the small coloured object/bar drawn inside it, and needs no
  hand-coded coordinates so it transfers to new videos.
* **Tactor** = the largest *saturated-green* blob **outside** the monitor (the
  monitor also renders green, so we mask it out first).
* **Motors** = the *circular spools* (Hough circles) in the bottom-left.

Regions are detected **once** per session (the rig does not move between pairs)
and reused, then cached to `analysis_output/`.

If the cache is missing the next cell builds it (slow on the 35k-frame pair_001;
uses `stride=2`).

In [ ]:
if not (OUT / 'sessions_meta.csv').exists():
    build(SESSION, stride=2, out_dir=OUT)

meta = pd.read_csv(OUT / 'sessions_meta.csv')
display(meta)

# Show the detected region layout.
import matplotlib.image as mpimg
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(mpimg.imread(OUT / 'regions_overlay.png'))
ax.set_title('Detected regions (vision=orange, motors=blue, tactor=green)')
ax.axis('off')
plt.show()

## 2. Signals extracted per region

Per frame (see `region_signals.py`):
* VISION: object centroid `obj_x/obj_y` (orange or blue square on black) and
  green `bar_fill`.
* TACTOR: `tactor_x/tactor_y` (yellow centre preferred, green body fallback).
* MOTORS: per-spool black-line angle via **PCA of dark pixels** - the same
  method as the dedicated motor rig (`vision_angle.SpoolAngleDetector`), tuned
  for the small (~20px) spools (`min_pixels=6`, full disc). Angles are
  continuously unwrapped to remove the line's 180-deg ambiguity.

Each frame carries the absolute `timestamp` taken from `tracking.csv` (frame i
<-> tracking row i, 1:1 in this rig), which is what lets us align video time
with the simulation and the motor log.

In [ ]:
rep = SR.build_summary(SESSION, OUT, fs=15.0)
summary, cmd_ack, log, pairs = rep['summary'], rep['cmd_ack'], rep['log'], rep['pairs']

# Peek at one pair's signal table.
example = pairs['pair_002']['signals']
print('signal columns:', list(example.columns))
display(example.head())

## 3. Tracking vs video - detection accuracy

The object the camera sees on the monitor is the *same* object the simulation
logged in `tracking.csv` (`object_x/object_y`). Comparing them validates the
**visual detection**: if the video-detected object reproduces the logged object
trajectory, detection is accurate.

Because the two live in different pixel frames (simulation screen px vs camera
panel px), we fit a best **affine** map before scoring the residual:
* `vision_accuracy_R2` - variance of the logged trajectory explained by the
  video trajectory after scale+offset (1.0 = perfect).
* `vision_detection_rate` - fraction of frames the object was found at all.

In [ ]:
# Overlay logged vs video object motion for one pair (normalised) to see the
# agreement directly.
p = 'pair_002'
sig = pairs[p]['signals']; trk = pairs[p]['tracking']

def _norm(a):
    a = np.asarray(a, float)
    return (a - np.nanmean(a)) / (np.nanstd(a) + 1e-9)

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(trk['t'], _norm(trk['object_x']), label='tracking object_x (sim)', lw=1)
ax.plot(sig['t'], _norm(sig['obj_x']), label='video object_x (camera)', lw=1, alpha=0.8)
ax.set_xlabel('time (s)'); ax.set_ylabel('z-scored x'); ax.legend(loc='upper right')
ax.set_title(f'{p} ({pairs[p]["tracking"]["finger"].iloc[0]}): logged vs video object position')
plt.tight_layout(); plt.show()

display(summary[['finger', 'pair', 'vision_detection_rate', 'vision_accuracy_R2',
                 'vision_accuracy_r', 'display_latency_ms', 'display_corr']].round(3))

## 4. Latency analysis (the actuation chain)

**Method.** For two time-series we resample both onto a common uniform grid,
take the **speed** (`|d/dt|`), z-score, and find the lag that maximises their
normalised cross-correlation. Speed (motion onset) is a sharper, frame-rate- and
frame-of-reference-invariant alignment feature than raw position. A positive lag
means the second signal *follows* the first. (`latency_analysis.estimate_lag`;
validated to recover a known 0.200 s shift exactly.)

Links reported:
* **hand -> vision**: tracked finger motion -> object on the monitor.
* **hand -> tactor**: tracked finger motion -> physical tactor motion (the full
  haptic actuation latency).
* **vision -> motor**: object on screen -> spool angle (same video clock, so a
  clean physical motor-response delay).

The `*_corr` column is the peak correlation; **treat a latency as unreliable
when its corr is low** (e.g. < ~0.3), which happens when a signal barely moves
(the tactor displacement here is only a few pixels).

**Orientation correction.** Each side-camera region is rotated relative to the
finger-tracking (top-camera) frame, defined as 0 deg: **vision = 180 deg**, **tactor (with finger) = 135 deg ccw** (90 deg at rest). We rotate the detected object/tactor into the finger frame (`session_report.ORIENTATION_DEG`) and report a `*_dir_corr` column = signed agreement of movement *direction* with the hand (+1 same way, -1 opposite), complementing the direction-blind magnitude latency.

**hand -> motor (per trial).** For the motor we use an onset-style measure: over the interaction window the finger pushes repeatedly and each spool turns in response; we cross-correlate finger motion vs each spool's angular motion (averaging the lag over all push cycles) and **average the 3 motors** (they are commanded together). This is one robust latency per trial (`hand_to_motor_latency_ms`), and succeeds where the on-screen-object->spool correlation was too weak.

In [ ]:
lat_cols = ['finger', 'pair',
            'hand_to_vision_latency_ms', 'hand_to_vision_corr',
            'hand_to_tactor_latency_ms', 'hand_to_tactor_corr',
            'hand_to_motor_latency_ms', 'hand_to_motor_corr']
display(summary[lat_cols].round(3))

# Visualise the cross-correlation curve for hand->tactor on one pair.
p = 'pair_002'; sig = pairs[p]['signals']; trk = pairs[p]['tracking']
hand = SR._motion_2d(trk['active_finger_x'], trk['active_finger_y'])
tac = SR._motion_2d(sig['tactor_x'], sig['tactor_y'])
res = LA.estimate_lag(trk['t'].to_numpy(), hand, sig['t'].to_numpy(), tac, fs=15.0, max_lag_s=2.0)
print(f'{p} hand->tactor: lag={res.lag_ms:+.0f} ms, peak corr={res.peak_corr:.2f}, n={res.n}')

## 5. Motor commands vs motor movement

`motor_commands.txt` is the ESP32 bridge log. We parse three streams
(`motor_commands.py`):
* `UDP_IN`  - position commanded by the backend,
* `SERIAL_OUT` - forwarded to the ESP32,
* `SERIAL_IN`  - the ESP32's `OK:` acknowledgement (applied position).

**Coverage gap.** The log window is *after* all four videos, so there is no
temporal overlap to cross-correlate commanded position against the video spool
angle. The latency we *can* measure is the **firmware command->ack** round-trip
(`UDP_IN` -> matching `SERIAL_IN`). The video spool angle is still useful as an
independent characterisation of motor motion (range, responsiveness).

**Spool/line detection.** The 3 motors are equal white discs stacked vertically, each with a dark groove. We locate the spool column by image *structure* (variance), place equal discs down the bright stack, and keep only discs that are real (high interior variance) - so a session showing only 2 spools yields 2, not a phantom 3rd. Each groove angle is measured by dark-pixel PCA.

In [ ]:
if log.n_lines == 0 or cmd_ack.empty:
    print('No motor-command log for this session (ESP32 bridge log absent).')
    print('=> motor latency metrics are N/A for this run; runs WITH a log get them.')
else:
    print('log lines=%d parsed=%d motors=%s' % (log.n_lines, log.n_parsed, log.motor_columns))
    print('coverage:', log.t_start, '..', log.t_end)
    for _, r in summary.iterrows():
        print('  %s (%s): overlap=%s' % (r['pair'], r['finger'], r['motor_log_overlap']))
    med = cmd_ack['latency_ms'].median(); p90 = cmd_ack['latency_ms'].quantile(0.9)
    print('firmware command->ack latency: median=%.1f ms, p90=%.1f ms, n=%d' % (med, p90, len(cmd_ack)))
    mcol = log.motor_columns[0]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].hist(cmd_ack['latency_ms'], bins=40, color='#0072B2')
    ax[0].set_title('Firmware command->ack latency'); ax[0].set_xlabel('ms')
    cmd = log.commanded(); ack = log.acknowledged()
    ax[1].plot(cmd['timestamp'], cmd[mcol], label='commanded ' + mcol, lw=1)
    ax[1].plot(ack['timestamp'], ack[mcol], label='applied ' + mcol, lw=1, alpha=0.8)
    ax[1].set_title('Motor ' + mcol + ': commanded vs applied'); ax[1].legend(); ax[1].tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()


## 6. Per-finger summary table

One row per finger with the accuracy and latency of **detection**, **tactor**,
and **motors**. Missing/uncomputable cells are `NaN`/`False` (never faked) -
notably the motor<->video latency, which needs an overlapping motor log.

In [ ]:
view = summary[SR.SUMMARY_VIEW].copy()
num = view.select_dtypes('number').columns
view[num] = view[num].round(3)
view.to_csv(OUT / 'per_finger_summary.csv', index=False)
print('saved', OUT / 'per_finger_summary.csv')
display(view)

### How to read the table
* **detection** - `vision_detection_rate` (frames the object was found),
  `vision_accuracy_R2` (how well the video object reproduces the logged object),
  `hand_to_vision_latency_ms` (finger -> on-screen object).
* **tactor** - `tactor_detection_rate`, `hand_to_tactor_corr` (reliability;
  low because tactor travel is only a few px), `hand_to_tactor_latency_ms`.
* **motors** - `motor_spool_detection_rate`, `vision_to_motor_latency_ms`
  (object -> spool, from video), `motor_cmd_to_ack_latency_ms` (firmware),
  `motor_log_overlap` (False everywhere here = the known data gap).

To extend to a new session: point `SESSION` at it; `build()` re-detects regions
and re-extracts. If that session's motor log overlaps the video, the
command->spool latency becomes computable automatically.